In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("SqlWarehouseDBUCostReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")
dbutils.widgets.text("workspace_ids", "", "Workspace IDs (comma-separated, blank=all)")

In [ ]:
# =======================================================
# SQL Warehouse DBU Cost Client
# =======================================================
# Sibling of PipelineDBUCostClient. Collects SQL warehouse (Classic / Pro /
# Serverless) DBU into the dbspend360_sql_warehouse_dbu_cost staging table.
# Differences vs the pipeline collector:
#   * usage filter:          billing_origin_product = 'SQL'
#                            AND usage_metadata.warehouse_id IS NOT NULL.
#                            The second predicate is NOT redundant: serverless
#                            JOBS (JOBS_SERVERLESS_COMPUTE) also bill under the
#                            SQL origin product but carry a 100%-NULL
#                            warehouse_id (~62K rows/month) - they belong to the
#                            jobs tabs, not here. Validated in plan Q1.
#   * price join:            unchanged from the pipeline collector - INNER (not
#                            LEFT) + the two-directional PRICE_JOIN_* guard. A
#                            DROP means a SKU lost its list price (silent
#                            undercount); a FAN_OUT means a SKU matched >1
#                            overlapping price row (silent cost inflation).
#                            Assert the join is exactly 1:1. See plan §5.4.
#   * aggregation key:       (warehouse_id, usage_date). warehouse_id is
#                            ACCOUNT-unique (unlike pipeline_id, which is only
#                            unique within a workspace), so workspace_id is
#                            carried as a coverage dimension but is NOT keyed.
#                            No cluster_id in the grain: validation confirmed
#                            every cluster_id is NULL for SQL warehouse SKUs
#                            (managed compute), and 0 warehouse-days carry >1
#                            cluster_id (plan Q2).
#   * warehouse_type:        derived from sku_name (SERVERLESS / PRO / CLASSIC)
#                            so the staging load stays a single-table scan - no
#                            system.compute.warehouses join here. Derived AFTER
#                            the aggregation from the collapsed SKU set so it is
#                            a deterministic function of the MERGE key (see the
#                            precedence note at the derivation below).
#   * no update/maintenance: there is no warehouse analogue of the DLT
#                            update / maintenance sub-run split.
#   * no compute_mode:       all three warehouse types run on Databricks-managed
#                            compute, so the serverless/classic distinction that
#                            drives the pipeline tab's cost_basis does not apply.
#                            There is no customer-account VM cost to be missing.
#   * MERGE key:             (warehouse_id, usage_date) - both NON-nullable, so
#                            plain '=' is correct. The null-safe '<=>' the
#                            pipeline collector needs is a cluster_id concern
#                            only, and cluster_id does not exist here.
class SqlWarehouseDBUCostClient:

    TABLE_NAME = "dbspend360_sql_warehouse_dbu_cost"

    def __init__(self, audit_table: str, target_table: str, covered_table: str, overlap_days: int, logger=None):
        self.audit_table = audit_table
        self.target_table = target_table
        self.covered_table = covered_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("SqlWarehouseDBUCostClient")
        raw_ws = dbutils.widgets.get("workspace_ids")
        if raw_ws.strip() == "":
            self.workspace_ids = None
        else:
            self.workspace_ids = [w.strip() for w in raw_ws.split(",") if w.strip()]

    def compute_and_merge_dbu_cost(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(f"Loading SQL warehouse DBU cost from {start_dt} to {end_dt}")

            # Primary filter: SQL-origin billing rows that actually carry a
            # warehouse_id. billing_origin_product = 'SQL' alone is NOT enough -
            # JOBS_SERVERLESS_COMPUTE shares that origin product with a NULL
            # warehouse_id (plan Q1), and those rows are serverless jobs.
            # Pre-project the usage side to clean, uniquely-named columns BEFORE
            # the join (all struct access resolved here, no qualified refs
            # survive into the join). u_sku_name is renamed so it never collides
            # with the price table's sku_name. On serverless Spark Connect a
            # column that participates in an equi-join key becomes unresolvable
            # when referenced (qualified) AFTER the join - even with no name
            # collision - so we carry sku through under u_sku_name and never
            # touch a join-key column post-join. See plan §5.4.
            usage_df = (
                spark.table("system.billing.usage")
                     .filter(
                         (F.col("usage_date") >= F.lit(start_dt)) &
                         (F.col("usage_date") <= F.lit(end_dt)) &
                         (F.col("billing_origin_product") == F.lit("SQL")) &
                         (F.col("usage_metadata")["warehouse_id"].isNotNull())
                     )
            )
            # Optional operator filter. workspace_ids narrows the SCAN only - it
            # is deliberately NOT part of the grain or the MERGE key, because
            # warehouse_id is account-unique.
            if self.workspace_ids is not None:
                usage_df = usage_df.filter(F.col("workspace_id").isin(self.workspace_ids))

            u = usage_df.select(
                F.col("usage_metadata")["warehouse_id"].alias("warehouse_id"),
                F.col("usage_date"),
                # Carried for coverage labelling (add_workspace_covered), never keyed.
                F.col("workspace_id"),
                F.col("sku_name").alias("u_sku_name"),
                F.col("usage_start_time"),
                F.col("usage_quantity"),
            )

            lp = spark.table("system.billing.list_prices").select(
                F.col("sku_name").alias("lp_sku_name"),
                F.col("price_start_time"),
                F.col("price_end_time"),
                F.col("pricing"),
            )

            # INNER join: a missing/renamed SKU must NOT silently vanish (a LEFT
            # join + NULL price would produce a NULL row_cost that SUM skips).
            # Every column has a unique name, so the condition and all
            # downstream references are unambiguous bare names.
            joined = u.join(
                lp,
                on=(
                    (F.col("u_sku_name") == F.col("lp_sku_name")) &
                    (F.col("usage_start_time") >= F.col("price_start_time")) &
                    (
                        (F.col("usage_start_time") < F.col("price_end_time")) |
                        F.col("price_end_time").isNull()
                    )
                ),
                how="inner",
            )

            # Two-directional 1:1 guard (plan §5.4). A DROP (join < left) means a
            # SKU had no list price and its usage silently vanished (SKU drift).
            # A FAN_OUT (join > left) means a SKU matched >1 overlapping price
            # row and its cost is silently MULTIPLIED. Checking only the drop
            # direction would miss the inflation case. Assert equality.
            #
            # Cost note (deliberate): these two .count() actions add two scans of
            # the filtered usage set purely for the 1:1 assertion; correctness
            # over speed is the intended trade-off. Cache the projected usage
            # set first.
            u = safe_cache(u)
            left_cnt = u.count()
            join_cnt = joined.count()
            if join_cnt != left_cnt:
                direction = "DROP" if join_cnt < left_cnt else "FAN_OUT"
                self.logger.warning(
                    "PRICE_JOIN_%s: usage rows %d -> joined rows %d (delta %+d). "
                    "Expected 1:1 (one list price per SKU/time). Investigate "
                    "before trusting totals.",
                    direction, left_cnt, join_cnt, join_cnt - left_cnt,
                )

            # warehouse_id is documented as ACCOUNT-unique, which is what lets
            # workspace_id sit outside the grain. If that ever breaks, the
            # aggregate below would silently pick ONE workspace per warehouse-day
            # and mislabel workspace_covered for the others - so surface it.
            # Runs against the cached projection, so it costs no extra scan of
            # system.billing.usage.
            ws_fanout = (
                u.select("warehouse_id", "usage_date", "workspace_id")
                 .distinct()
                 .groupBy("warehouse_id", "usage_date")
                 .agg(F.countDistinct("workspace_id").alias("n_workspaces"))
                 .filter(F.col("n_workspaces") > 1)
            )
            if ws_fanout.limit(1).count() > 0:
                self.logger.warning(
                    "WAREHOUSE_WORKSPACE_FANOUT: %d (warehouse_id, usage_date) pairs "
                    "span >1 workspace_id. warehouse_id is expected to be "
                    "account-unique; workspace_id is picked deterministically "
                    "(max) per warehouse-day, so workspace_covered may be "
                    "mislabeled for the non-winning workspace(s).",
                    ws_fanout.count(),
                )

            # All columns are already clean, uniquely-named, and struct-free.
            base = joined.select(
                F.col("warehouse_id"),
                F.col("usage_date"),
                F.col("workspace_id"),
                F.col("u_sku_name").alias("sku_name"),
                F.col("usage_quantity"),
                F.col("pricing")["default"].cast("double").alias("price"),
            )

            with_cols = base.withColumn("row_cost", F.col("usage_quantity") * F.col("price"))

            agg_df = (
                with_cols
                .groupBy("warehouse_id", "usage_date")
                .agg(
                    F.sum("row_cost").alias("databricks_cost"),
                    F.concat_ws(
                        " + ",
                        F.array_sort(F.collect_set("sku_name")),
                    ).alias("sku_name"),
                    # max() not first(): first() is non-deterministic on an
                    # unordered group. In practice a warehouse lives in exactly
                    # one workspace, so the two agree; max() just guarantees the
                    # same answer on every re-run of an overlapping window.
                    F.max("workspace_id").alias("workspace_id"),
                )
                # warehouse_type from the COLLAPSED SKU set rather than a
                # per-row value aggregated with first(). Applying the LIKE rules
                # to the concatenated set gives a fixed precedence
                # (SERVERLESS > PRO > CLASSIC) and therefore a deterministic
                # function of the MERGE key even in the rare case where a
                # warehouse-day carries more than one SKU (e.g. a type change
                # mid-day). .like is case-sensitive, hence the upper().
                .withColumn(
                    "warehouse_type",
                    F.when(F.upper(F.col("sku_name")).like("%SERVERLESS%"), F.lit("SERVERLESS"))
                     .when(F.upper(F.col("sku_name")).like("%PRO%"), F.lit("PRO"))
                     .otherwise(F.lit("CLASSIC")),
                )
                .withColumn("currency", F.lit("USD"))
            )

            agg_df = add_workspace_covered(agg_df, self.covered_table, "workspace_id")

            dbu_inc_df = agg_df.select(
                "warehouse_id",
                "usage_date",
                "databricks_cost",
                "currency",
                "sku_name",
                "warehouse_type",
                "workspace_id",
                "workspace_covered",
            )

            if dbu_inc_df.limit(1).count() == 0:
                # Empty window is a legitimate state (no SQL warehouse usage in
                # the period). Don't fail - log an explanatory INFO and write a
                # SUCCESS audit row with row_count=0 below.
                self.logger.info(
                    "No SQL warehouse DBU rows after filtering / aggregation. "
                    "Verify there exists system.billing.usage data with "
                    "billing_origin_product = 'SQL' AND "
                    "usage_metadata.warehouse_id IS NOT NULL in the window."
                )
                merged_row_count = 0
            else:
                dbu_inc_df = (
                    dbu_inc_df
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )
                dbu_inc_df = safe_cache(dbu_inc_df)

                merged_row_count = dbu_inc_df.count()

                validate_source_schema(
                    dbu_inc_df,
                    {"warehouse_id": "string", "usage_date": "date",
                     "databricks_cost": "double", "warehouse_type": "string",
                     "workspace_id": "string"},
                    self.target_table, self.logger,
                )
                validate_no_negative_costs(
                    dbu_inc_df, ["databricks_cost"], self.target_table, self.logger,
                )
                validate_currency_consistency(dbu_inc_df, "currency", self.target_table, self.logger)

                ensure_boolean_columns(self.target_table, ["workspace_covered"], logger=self.logger)
                target = DeltaTable.forName(spark, self.target_table)
                (target.alias("t")
                    .merge(
                        dbu_inc_df.alias("s"),
                        # Both key columns are NON-nullable, so plain '=' is
                        # correct here - the null-safe '<=>' the pipeline
                        # collector needs guards a nullable cluster_id, which
                        # this table does not carry (plan Q2).
                        "t.warehouse_id = s.warehouse_id "
                        "AND t.usage_date = s.usage_date",
                    )
                    .whenMatchedUpdate(set={
                        "databricks_cost": "s.databricks_cost",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "warehouse_type": "s.warehouse_type",
                        "workspace_id": "s.workspace_id",
                        "workspace_covered": "s.workspace_covered",
                        "updated_at": "current_timestamp()",
                    })
                    .whenNotMatchedInsert(values={
                        "warehouse_id": "s.warehouse_id",
                        "usage_date": "s.usage_date",
                        "databricks_cost": "s.databricks_cost",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "warehouse_type": "s.warehouse_type",
                        "workspace_id": "s.workspace_id",
                        "workspace_covered": "s.workspace_covered",
                        "created_at": "current_timestamp()",
                        "updated_at": "current_timestamp()",
                    })
                    .execute()
                )

                safe_unpersist(dbu_inc_df)
                get_merge_metrics(self.target_table, self.logger)

                validate_post_merge(
                    self.target_table, "usage_date",
                    start_dt, end_dt, merged_row_count, self.logger,
                )

            safe_unpersist(u)

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", merged_row_count, "")
            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class SqlWarehouseDBUCostReporterApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        target_table = build_table_fqn(catalog, schema, "dbspend360_sql_warehouse_dbu_cost")

        self.client = SqlWarehouseDBUCostClient(
            audit_table=audit_table,
            target_table=target_table,
            covered_table=build_table_fqn(catalog, schema, "dbspend360_covered_workspaces"),
            overlap_days=overlap_days,
            logger=logger,
        )

    def run(self):
        self.client.compute_and_merge_dbu_cost()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = SqlWarehouseDBUCostReporterApp()
app.run()